# Comparación de fine-tuning con distintos prompts

Compara el modelo base y adaptadores LoRA entrenados con distintos prompts (por ejemplo, zero/one/few-shot).

Métricas: EM, F1, F1 por preguntas respondibles/no respondibles, tasa de respuesta vacía, latencia media/p50/p95, tokens/s, errores y pico de VRAM. El tiempo de entrenamiento no se mide.

**Ejecuta Jupyter desde la raíz del repositorio.** Revisa y corrige las rutas de datos y adaptadores en la celda siguiente antes de ejecutar. Este notebook no modifica los scripts del proyecto; guarda resultados en `outputs/prompt_comparison/`.


In [ ]:
from pathlib import Path
import time, re, gc
import numpy as np
import pandas as pd
import torch

PROJECT_ROOT = Path.cwd()
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "prompt_comparison"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# CAMBIA estas dos rutas por las rutas reales de los archivos usados por el preprocesado.
QUESTIONS_PATH = PROJECT_ROOT / "data" / "questions.csv"
CONTEXTS_PATH = PROJECT_ROOT / "data" / "contexts.csv"

MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"
ADAPTERS = {
    "FT_zero": PROJECT_ROOT / "models" / "prompt_training" / "qwen_lora_zero",
    "FT_one":  PROJECT_ROOT / "models" / "prompt_training" / "qwen_lora_one",
    "FT_few":  PROJECT_ROOT / "models" / "prompt_training" / "qwen_lora_few",
}
RUN_BASE_MODEL = True
MAX_NEW_TOKENS = 96
DO_SAMPLE = False
WARMUP_QUESTIONS = 2
LIMIT_TEST = None  # poner 10 para probar rápidamente; None evalúa todo el test
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Misma plantilla de evaluación para todos los adaptadores.
EVAL_TEMPLATE = """Answer the question using only the provided context.
If the context does not contain the answer, say that you cannot answer.
Return only the answer, without explanations.

Context:
{context}

Question: {question}
Answer:"""

print("Proyecto:", PROJECT_ROOT, "| Dispositivo:", DEVICE)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


## 1. Cargar datos

Utiliza la función existente del proyecto. Si los nombres o rutas de los ficheros son distintos, corrige solo las variables de configuración de arriba.

In [ ]:
from src.preprocess.preprocess import prepare_dataset

for path in (QUESTIONS_PATH, CONTEXTS_PATH):
    if not path.exists():
        raise FileNotFoundError(f"No encuentro {path}. Corrige la ruta en la celda de configuración.")

dataset = prepare_dataset(str(QUESTIONS_PATH), str(CONTEXTS_PATH))
df = pd.DataFrame(dataset)
required = {"question", "answer", "context"}
missing = required - set(df.columns)
if missing:
    raise ValueError(f"Faltan columnas {missing}; columnas disponibles: {list(df.columns)}")
if "has_answer" not in df.columns:
    df["has_answer"] = df["answer"].fillna("").astype(str).str.strip().ne("")
df = df.reset_index(drop=True)
if LIMIT_TEST is not None:
    df = df.iloc[:LIMIT_TEST].copy()
print(f"Ejemplos: {len(df)}")
display(df.head(3))


## 2. Métricas y carga del modelo

La latencia se mide solo durante la generación, después de la carga y el calentamiento. Los contextos ya están preparados, así que el tiempo de recuperación no se incluye.

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

def normalize_answer(text):
    text = str(text or "").lower()
    text = re.sub(r"[^\\w\\s]", " ", text, flags=re.UNICODE)
    text = re.sub(r"\\b(a|an|the)\\b", " ", text)
    return " ".join(text.split())

def exact_match(pred, gold):
    return float(normalize_answer(pred) == normalize_answer(gold))

def token_f1(pred, gold):
    pt, gt = normalize_answer(pred).split(), normalize_answer(gold).split()
    if not pt and not gt: return 1.0
    if not pt or not gt: return 0.0
    common = {}
    for t in pt: common[t] = common.get(t, 0) + 1
    overlap = 0
    for t in gt:
        if common.get(t, 0):
            overlap += 1
            common[t] -= 1
    if not overlap: return 0.0
    p, r = overlap / len(pt), overlap / len(gt)
    return 2 * p * r / (p + r)

def answerable(row):
    x = row.get("has_answer", False)
    if isinstance(x, str):
        return x.strip().lower() in {"true", "1", "yes", "si", "sí"}
    return bool(x)

def make_prompt(row):
    return EVAL_TEMPLATE.format(
        context=str(row.get("context", "") or ""),
        question=str(row.get("question", "") or "")
    )

def sync_device():
    if torch.cuda.is_available():
        torch.cuda.synchronize()

def load_model_and_tokenizer(adapter_path=None):
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, use_fast=True)
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token = tokenizer.eos_token
    dtype = torch.float16 if torch.cuda.is_available() else torch.float32
    model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=dtype)
    model.to(DEVICE)
    if adapter_path is not None:
        adapter_path = Path(adapter_path)
        if not (adapter_path / "adapter_config.json").exists():
            raise FileNotFoundError(f"No parece un adaptador PEFT válido: {adapter_path}")
        model = PeftModel.from_pretrained(model, str(adapter_path))
    model.eval()
    return model, tokenizer

@torch.inference_mode()
def generate_one(model, tokenizer, row):
    prompt = make_prompt(row)
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=2048)
    inputs = {k: v.to(DEVICE) for k, v in inputs.items()}
    kwargs = {
        "max_new_tokens": MAX_NEW_TOKENS,
        "do_sample": DO_SAMPLE,
        "pad_token_id": tokenizer.pad_token_id,
        "eos_token_id": tokenizer.eos_token_id,
    }
    sync_device()
    t0 = time.perf_counter()
    output = model.generate(**inputs, **kwargs)
    sync_device()
    elapsed = time.perf_counter() - t0
    new_ids = output[0][inputs["input_ids"].shape[1]:]
    pred = tokenizer.decode(new_ids, skip_special_tokens=True).strip()
    n_tokens = int(new_ids.shape[0])
    return pred, elapsed, n_tokens, int(inputs["input_ids"].shape[1])

print("Funciones listas.")


## 3. Ejecutar la comparación

El notebook vuelve a cargar el modelo base para cada adaptador, para evitar contaminación entre variantes. Si no tienes alguno de los adaptadores configurados, quítalo del diccionario `ADAPTERS`.

In [ ]:
def evaluate_variant(name, adapter_path):
    if adapter_path is not None:
        adapter_path = Path(adapter_path)
        if not adapter_path.exists():
            raise FileNotFoundError(f"Adaptador no encontrado ({name}): {adapter_path}")
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()
    t_load = time.perf_counter()
    model, tokenizer = load_model_and_tokenizer(adapter_path)
    load_seconds = time.perf_counter() - t_load

    for i in range(min(WARMUP_QUESTIONS, len(df))):
        generate_one(model, tokenizer, df.iloc[i])

    rows = []
    for idx, row in df.iterrows():
        try:
            pred, latency, n_tokens, n_input = generate_one(model, tokenizer, row)
            rows.append({
                "variant": name, "index": int(idx),
                "question": str(row["question"]), "gold": str(row["answer"] or ""),
                "prediction": pred, "has_answer": answerable(row),
                "em": exact_match(pred, row["answer"]),
                "f1": token_f1(pred, row["answer"]),
                "latency_seconds": latency, "generated_tokens": n_tokens,
                "input_tokens": n_input,
                "tokens_per_second": n_tokens / latency if latency > 0 else np.nan,
                "empty_prediction": not bool(pred.strip()), "error": None,
            })
        except Exception as exc:
            rows.append({
                "variant": name, "index": int(idx),
                "question": str(row.get("question", "")), "gold": str(row.get("answer", "")),
                "prediction": "", "has_answer": answerable(row), "em": 0.0, "f1": 0.0,
                "latency_seconds": np.nan, "generated_tokens": 0, "input_tokens": 0,
                "tokens_per_second": np.nan, "empty_prediction": True, "error": repr(exc),
            })

    out = pd.DataFrame(rows)
    valid = out[out["error"].isna()]
    ans = valid[valid["has_answer"] == True]
    unans = valid[valid["has_answer"] == False]
    peak_vram = torch.cuda.max_memory_allocated() / 1024**3 if torch.cuda.is_available() else np.nan
    lat = valid["latency_seconds"].dropna()
    summary = {
        "variant": name, "adapter_path": str(adapter_path) if adapter_path else "BASE",
        "n_total": len(out), "n_success": len(valid), "n_errors": int(out["error"].notna().sum()),
        "EM": valid["em"].mean(), "F1": valid["f1"].mean(),
        "answerable_F1": ans["f1"].mean() if len(ans) else np.nan,
        "unanswerable_F1": unans["f1"].mean() if len(unans) else np.nan,
        "empty_prediction_rate": valid["empty_prediction"].mean(),
        "total_generation_seconds": lat.sum(),
        "mean_latency_seconds": lat.mean(), "median_latency_p50_seconds": lat.median(),
        "p95_latency_seconds": lat.quantile(.95),
        "mean_tokens_per_second": valid["tokens_per_second"].mean(),
        "mean_generated_tokens": valid["generated_tokens"].mean(),
        "model_load_seconds_secondary": load_seconds, "peak_vram_gb": peak_vram,
    }
    del model, tokenizer
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return summary, out

variants = {"BASE": None} if RUN_BASE_MODEL else {}
variants.update(ADAPTERS)
summaries, prediction_tables = [], []

for name, path in variants.items():
    print(f"\nEvaluando {name}...")
    summary, predictions = evaluate_variant(name, path)
    summaries.append(summary)
    prediction_tables.append(predictions)
    print(pd.Series(summary).to_string())

summary_df = pd.DataFrame(summaries)
predictions_df = pd.concat(prediction_tables, ignore_index=True)


## 4. Resultados, gráficos y exportación

In [ ]:
display(summary_df.sort_values(["F1", "mean_latency_seconds"], ascending=[False, True]).reset_index(drop=True))

summary_path = OUTPUT_DIR / "summary_metrics.csv"
predictions_path = OUTPUT_DIR / "per_question_predictions.csv"
summary_df.to_csv(summary_path, index=False)
predictions_df.to_csv(predictions_path, index=False)
print("Guardado:", summary_path)
print("Guardado:", predictions_path)


In [ ]:
import matplotlib.pyplot as plt

plot_df = summary_df.dropna(subset=["F1", "mean_latency_seconds"])
if len(plot_df):
    plt.figure()
    plt.scatter(plot_df["mean_latency_seconds"], plot_df["F1"])
    for _, row in plot_df.iterrows():
        plt.annotate(row["variant"], (row["mean_latency_seconds"], row["F1"]))
    plt.xlabel("Latencia media por pregunta (s)")
    plt.ylabel("F1")
    plt.title("Calidad frente a velocidad")
    plt.grid(True)
    plt.show()

    plt.figure()
    plt.bar(plot_df["variant"], plot_df["mean_latency_seconds"])
    plt.ylabel("Latencia media (s)")
    plt.title("Latencia por variante")
    plt.xticks(rotation=30, ha="right")
    plt.tight_layout()
    plt.show()


## 5. Inspección de errores

La métrica agregada no cuenta toda la historia. Inspecciona ejemplos concretos y, especialmente, los errores de abstención. Esta versión marca respuestas vacías, pero no clasifica automáticamente frases como “I cannot answer” como abstenciones; eso puede añadirse cuando fijemos el formato de salida.

In [ ]:
display(
    predictions_df.pivot_table(
        index=["index", "question", "gold", "has_answer"],
        columns="variant", values="prediction", aggfunc="first"
    ).reset_index().head(30)
)

errors_df = predictions_df[(predictions_df["em"] == 0) | predictions_df["error"].notna()]
display(errors_df[[
    "variant", "index", "question", "gold", "prediction", "f1",
    "latency_seconds", "error"
]].head(50))


## Limitaciones que hay que tener presentes

- Las rutas `QUESTIONS_PATH` y `CONTEXTS_PATH` son ejemplos de configuración y hay que ajustarlas a los ficheros reales del proyecto.
- Este notebook usa una **misma plantilla de evaluación para todos**. Compara qué adaptador generaliza mejor con una entrada común; no evalúa automáticamente cada adaptador con su plantilla de entrenamiento original.
- La F1 implementada es token-level F1 de QA con normalización básica. Antes de comparar con métricas históricas, comprueba que coincide con la definición que usaban los scripts anteriores.
- La latencia no incluye carga del modelo ni recuperación RAG. El tiempo de carga se muestra aparte y el entrenamiento no se mide.
- Para una comparación rigurosa, usa el mismo split de test, los mismos contextos y parámetros de generación, y revisa las predicciones por pregunta.
